In [1]:
import os
import numpy as np
import pandas as pd
import glob

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Update these paths to match your local setup
RAW_DATA_PATH = r"../../data/raw/multi_class_pothole"
MANUAL_DATA_PATH = r"../../data/raw/binary_mannual_collection/Yashobhoomi.csv"
OUTPUT_CSV_PATH = r"../../data/combined/multi_class_pothole/combined.csv"

# Physics Constants
GRAVITY = 9.80665
TARGET_FREQ = 10  # Hz
WINDOW_SECONDS = 2
WINDOW_SIZE = int(WINDOW_SECONDS * TARGET_FREQ) # 20 rows per window
STRIDE = 5  # Slide window by 0.5s (Data Augmentation)

# ==========================================
# UPDATED CLASS MAPPING
# 0: Good
# 1: Vibrations
# 2: Small/Medium Pothole (Combined)
# 3: Big Pothole
# ==========================================

# File Rules Mapping (Updated Indices)
FILE_RULES = {
    # Old Class 4 (Big) -> New Class 3
    "both bad (2).csv":        [(63, -1, 3)],
    "both bad (3).csv":        [(22, 420, 3), (608, -1, 3)],
    "both bad.csv":            [(0, -1, 3)],
    "right bad.csv":           [(0, -1, 3)],
    
    # Old Class 3 (Medium) -> New Class 2
    "both medium (2).csv":     [(270, -1, 2)],
    "Both medium.csv":         [(250, 450, 2)],
    "left medium.csv":         [(990, -1, 2)],
    "left_medium_right.csv":   [(140, 420, 2)],

    # Old Class 2 (Small) -> New Class 2 (Unchanged index, but merged concept)
    "both small.csv":          [(340, -1, 2)],
    "right small (2).csv":     [(0, -1, 2)],
    
    # Mixed Files (Updated specifically)
    "right small.csv":         [(0, 900, 0), (900, 1120, 2)],
    "vibrations_small_medium.csv": [(0, 520, 1), (520, 910, 2), (910, -1, 3)], # Note: last tuple was 4->3
    
    # Old Class 1 (Vibration) -> New Class 1 (Unchanged)
    "both vibration.csv":      [(0, -1, 1)]
}

# Yashobhoomi Coordinates for Good Road (0)
GOOD_SEGMENTS_COORDS = [
    (28.408258, 76.927417, 28.405190, 76.919277),
    (28.517245, 77.022301, 28.463858, 76.968626)
]

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_segment_indices(df, lat1, lon1, lat2, lon2):
    """Finds start/end row indices based on GPS coordinates"""
    dist_start = (df['latitude'] - lat1)**2 + (df['longitude'] - lon1)**2
    dist_end = (df['latitude'] - lat2)**2 + (df['longitude'] - lon2)**2
    return sorted([dist_start.idxmin(), dist_end.idxmin()])

def generate_synthetic_speed(length, base_speed_type):
    """Generates a speed array with noise."""
    if base_speed_type == 'fast': target = np.random.uniform(10, 20)
    elif base_speed_type == 'slow': target = np.random.uniform(2, 8)
    else: target = np.random.uniform(2, 20)
    
    noise = np.random.normal(0, 0.1, length)
    speed_profile = np.cumsum(noise) + target
    return np.clip(speed_profile, 0.5, 30)

def smooth_transition(arr1, arr2, overlap=5):
    """Linearly blends two arrays to prevent jerk at stitch point"""
    if len(arr1) < overlap or len(arr2) < overlap: 
        return np.vstack([arr1, arr2])
    
    part1 = arr1[:-overlap]
    part2 = arr2[overlap:]
    # Linear blend: (1-alpha)*A + alpha*B
    alpha = np.linspace(0, 1, overlap*2).reshape(-1, 1)
    
    # Simple average of the overlap region
    mid = (arr1[-overlap:] + arr2[:overlap]) / 2 
    # Note: To do a true linear blend you'd use weighted sum, 
    # but average is sufficient for quick stitching here.
    return np.vstack([part1, mid, part2])

# ==========================================
# 3. DATA LOADING
# ==========================================
def load_data_pool():
    pool = [] 
    print("Loading raw files...")
    
    # --- PART A: Process 100Hz Files ---
    files = sorted(glob.glob(os.path.join(RAW_DATA_PATH, "*.csv")))
    for fname in files:
        basename = os.path.basename(fname)
        if basename not in FILE_RULES: continue
            
        df = pd.read_csv(fname)
        if 'az' in df.columns: df['az'] += GRAVITY
            
        for start, end, label in FILE_RULES[basename]:
            if end == -1: segment = df.iloc[start:].copy()
            else: segment = df.iloc[start:end].copy()
                
            # Decimate 100Hz -> 10Hz
            segment = segment.iloc[::10]
            
            # Select columns
            data = segment[['ax', 'ay', 'az', 'wx', 'wy', 'wz']].values
            if len(data) > WINDOW_SIZE:
                pool.append({'data': data, 'label': label})

    # --- PART B: Process Yashobhoomi (10Hz) ---
    if os.path.exists(MANUAL_DATA_PATH):
        df_yash = pd.read_csv(MANUAL_DATA_PATH)
        for lat1, lon1, lat2, lon2 in GOOD_SEGMENTS_COORDS:
            s, e = get_segment_indices(df_yash, lat1, lon1, lat2, lon2)
            segment = df_yash.iloc[s:e]
            data = segment[['ax', 'ay', 'az', 'wx', 'wy', 'wz']].values
            if len(data) > WINDOW_SIZE:
                pool.append({'data': data, 'label': 0}) # Label 0 for Good

    return pool

# ==========================================
# 4. SYNTHESIS & CSV GENERATION
# ==========================================
def generate_csv():
    pool = load_data_pool()
    if not pool:
        print("Error: No data loaded. Check paths.")
        return

    output_rows = []
    window_counter = 0

    print("Synthesizing Windows...")
    
    # We will generate a fixed number of windows (e.g., 2000 windows)
    # by randomly stitching and sliding
    
    NUM_SCENARIOS = 500 # Number of "Stitched Roads" to create
    
    for _ in range(NUM_SCENARIOS):
        # 1. Pick Segments (Good -> Random Bad -> Good)
        good_segs = [p for p in pool if p['label'] == 0]
        bad_segs = [p for p in pool if p['label'] > 0]
        
        if not good_segs or not bad_segs: break
            
        seg_a = good_segs[np.random.randint(len(good_segs))]
        seg_b = bad_segs[np.random.randint(len(bad_segs))]
        
        # 2. Extract Chunks (3-6 seconds each)
        chunk_len = np.random.randint(30, 60)
        
        # Safety checks for index bounds
        idx_a = np.random.randint(0, max(1, len(seg_a['data'])-chunk_len))
        idx_b = np.random.randint(0, max(1, len(seg_b['data'])-chunk_len))
        
        clip_a = seg_a['data'][idx_a : idx_a+chunk_len]
        clip_b = seg_b['data'][idx_b : idx_b+chunk_len]
        
        # 3. Stitch & Smooth
        # Combine [Good, Bad, Good] or just [Good, Bad] to vary it
        raw_seq = smooth_transition(clip_a, clip_b)
        
        # Create Labels (matching length)
        # Note: Logic is simplified here. The transition zone gets the 'Bad' label
        lbls_a = np.zeros(len(clip_a)-5) # minus overlap
        lbls_b = np.full(len(clip_b), seg_b['label']) # approximate
        
        # Re-construct labels array to match raw_seq length
        # (This is a quick hack to match lengths; for perfect physics, track indices)
        
        combined_labels = np.concatenate([np.zeros(len(clip_a)), np.full(len(clip_b), seg_b['label'])])
        # Trim/Pad to match smoothed signal
        if len(combined_labels) > len(raw_seq): combined_labels = combined_labels[:len(raw_seq)]
        else: combined_labels = np.pad(combined_labels, (0, len(raw_seq)-len(combined_labels)), 'edge')

        # 4. Inject Speed
        scenario = np.random.choice(['fast_bad', 'slow_good', 'normal'])
        spd_type = 'fast' if scenario == 'fast_bad' else ('slow' if scenario == 'slow_good' else 'random')
        speed_col = generate_synthetic_speed(len(raw_seq), spd_type)
        
        # 5. Sliding Window & Save to List
        num_windows = (len(raw_seq) - WINDOW_SIZE) // STRIDE + 1
        
        for i in range(num_windows):
            start = i * STRIDE
            end = start + WINDOW_SIZE
            
            # The Data
            win_data = raw_seq[start:end] # Shape (20, 6)
            win_speed = speed_col[start:end]
            win_labels = combined_labels[start:end]
            
            # ---- CRITICAL FIX: skip incomplete windows ----
            if win_data.shape[0] != WINDOW_SIZE:
                continue

            # The Window Label (Max severity in this 2s window)
            # This works because 3 (Big) > 2 (Small/Med) > 1 (Vib) > 0 (Good)
            final_label = int(np.max(win_labels))
            
            # Create Rows
            for row_idx in range(WINDOW_SIZE):
                output_rows.append({
                    'window_id': window_counter,
                    'step': row_idx,
                    'ax': win_data[row_idx, 0],
                    'ay': win_data[row_idx, 1],
                    'az': win_data[row_idx, 2],
                    'wx': win_data[row_idx, 3],
                    'wy': win_data[row_idx, 4],
                    'wz': win_data[row_idx, 5],
                    'speed': win_speed[row_idx],
                    'label': final_label # The label is constant for the whole window
                })
            
            window_counter += 1

    # ==========================================
    # 5. WRITE TO CSV
    # ==========================================
    print(f"Writing {len(output_rows)} rows to {OUTPUT_CSV_PATH}...")
    
    # Ensure directory exists
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)
    
    df_out = pd.DataFrame(output_rows)
    df_out.to_csv(OUTPUT_CSV_PATH, index=False)
    print("Done!")

if __name__ == "__main__":
    generate_csv()

Loading raw files...
Synthesizing Windows...
Writing 116220 rows to ../../data/combined/multi_class_pothole/combined.csv...
Done!
